# Explicit differentiation of transient elastodynamics

In this tutorial, we reconstruct the axial distribution of Young's modulus in a three-dimensional elastic waveguide from transient displacement observations. The displacement trajectory is computed using a lumped-mass finite element discretization and the explicit central-difference method. A uniform material is first used to verify the forward solution against the first longitudinal mode. The heterogeneous inverse problem assigns different Young's modulus values to each axial element layer. Gradients of the observation loss are obtained by reverse-mode automatic differentiation (AD) through the time integration steps, while gradient checkpointing is used to trade recomputation for reduced memory. The implementation uses [JAX](https://github.com/jax-ml/jax) and [JAX-FEM](https://github.com/deepmodeling/jax-fem).

## Problem formulations

### Forward problem

Consider the free vibration of a three-dimensional elastic waveguide $\Omega=[0,L_x]\times[0,L_y]\times[0,L_z]$. The small-strain elastodynamic equations are

$$
\rho\ddot{\boldsymbol u}=\nabla\cdot\boldsymbol\sigma,
\qquad
\boldsymbol\sigma=\lambda\operatorname{tr}(\boldsymbol\varepsilon)\boldsymbol I+2\mu\boldsymbol\varepsilon,
\qquad
\boldsymbol\varepsilon=\frac{1}{2}\left(\nabla\boldsymbol u+\nabla\boldsymbol u^\top\right),
$$

where $\rho$ is the mass density, $\boldsymbol u$ is the displacement vector, $\boldsymbol\sigma$ is the Cauchy stress, $\boldsymbol\varepsilon$ is the strain tensor, and $\boldsymbol I$ is the identity tensor. The Lamé parameters $\lambda$ and $\mu$ are determined by the Young's modulus $E$ and the Poisson's ratio $\nu$. The left end satisfies $u_x=0$, while the normal transverse displacement is constrained on the lateral faces. The right end is traction-free. The waveguide is released from the initial state

$$
u_x(\boldsymbol x,0)=A\sin(kx),
\qquad
\dot{\boldsymbol u}(\boldsymbol x,0)=\boldsymbol 0,
\qquad
k=\frac{\pi}{2L_x}.
$$

The waveguide is released from rest, and the initial elastic strain energy is periodically exchanged with kinetic energy during the resulting undamped free vibration.

For a uniform $E$, the first fixed-free longitudinal mode has the exact solution

$$
u_x(\boldsymbol x,t)=A\sin(kx)\cos(c_p k t),
\qquad
c_p=\sqrt{\frac{E(1-\nu)}{(1+\nu)(1-2\nu)\rho}}.
$$

Let $\delta\boldsymbol u$ be an admissible virtual displacement and $\delta\boldsymbol\varepsilon=\frac{1}{2}(\nabla\delta\boldsymbol u+\nabla\delta\boldsymbol u^\top)$ the virtual strain. The weak form is

$$
\int_\Omega \rho\,\delta\boldsymbol u\cdot\ddot{\boldsymbol u}\,\mathrm d\Omega
+\int_\Omega \delta\boldsymbol\varepsilon:\boldsymbol\sigma\,\mathrm d\Omega=0.
$$

Using the finite element approximation $\boldsymbol u^h=\sum_a N_a\boldsymbol U_a$, where $N_a$ is the nodal shape function and $\boldsymbol U_a$ is the nodal displacement vector, gives the semi-discrete system

$$
\boldsymbol M\ddot{\boldsymbol U}+\boldsymbol F_{\mathrm{int}}=\boldsymbol 0,
$$

where the scalar mass coefficient, the corresponding matrix block, and the nodal internal force are

$$
m_{ab}=\int_\Omega \rho N_aN_b\,\mathrm d\Omega,\qquad
\boldsymbol M_{ab}=m_{ab}\boldsymbol I,\qquad
\boldsymbol F_{\mathrm{int},a}=\int_\Omega \boldsymbol B_a^\top\boldsymbol\sigma\,\mathrm d\Omega.
$$

Here $\boldsymbol B_a$ is the strain-displacement operator associated with node $a$. Row-sum lumping replaces the consistent mass matrix by the diagonal matrix

$$
\boldsymbol M_L=\operatorname{diag}(m_1\boldsymbol I,\ldots,m_{N_n}\boldsymbol I),\qquad
m_a=\sum_bm_{ab}=\int_\Omega \rho N_a\,\mathrm d\Omega.
$$

Here $N_n$ is the total number of finite element nodes. The acceleration at time step $n$ is therefore obtained without solving a linear system,

$$
\boldsymbol A^n=-\boldsymbol M_L^{-1}\boldsymbol F_{\mathrm{int}}^n.
$$

For a time-step size $\Delta t$, the fictitious previous state is initialized to second-order accuracy,

$$
\boldsymbol U^{-1}=\boldsymbol U^0-\Delta t\dot{\boldsymbol U}^0+\frac{1}{2}\Delta t^2\boldsymbol A^0.
$$

The central-difference displacement update is then

$$
\boldsymbol U^{n+1}=2\boldsymbol U^n-\boldsymbol U^{n-1}-\Delta t^2\boldsymbol M_L^{-1}\boldsymbol F_{\mathrm{int}}^n.
$$

### Inverse problem

The waveguide is divided into $N_x$ axial element layers, with different Young's modulus values assigned to each layer, $\boldsymbol E=(E_1,\ldots,E_{N_x})$. Let $\widehat{u}_r^n$ denote the observed axial displacement at probe $r$ and time step $n$. The modulus distribution is identified from

$$
\boldsymbol E_{\mathrm{id}}=\argmin_{\boldsymbol E}\frac{1}{N_o(N+1)}\sum_{r=1}^{N_o}\sum_{n=0}^{N}\left[\frac{u_r^n(\boldsymbol E)-\widehat{u}_r^n}{u_{\mathrm{rms}}}\right]^2.
$$

Here $N_o$ is the number of observation points, $N$ is the number of time steps, $u_r^n$ is the predicted axial displacement at probe $r$, $u_{\mathrm{rms}}$ is the root-mean-square observed displacement. The target modulus varies linearly along the waveguide,

$$
E^\star(x)=E_0\left(1-0.4\frac{x}{L_x}\right).
$$

where $E_0$ is the reference modulus. The axial displacement is observed at the center of each cross-section,

$$
\boldsymbol x_r=\left(\frac{rL_x}{N_x},\frac{L_y}{2},\frac{L_z}{2}\right),\qquad r=1,\ldots,N_x,
$$

so that $N_o=N_x$.

### Explicit differentiation and checkpointing

The central-difference scheme defines the explicit recurrence $\boldsymbol S^{n+1}=\mathcal G(\boldsymbol S^n,\boldsymbol E)$, where $\boldsymbol S^n=(\boldsymbol U^n,\boldsymbol U^{n-1})$ contains the states required by the next time step. Reverse-mode AD computes $\nabla_{\boldsymbol E}\mathcal L$ by backpropagating through this recurrence. Without checkpointing, the reverse pass retains intermediate values from the complete trajectory. Wrapping a block of time steps with `jax.checkpoint` allows its internal values to be discarded and recomputed from the block input during backpropagation. Shorter blocks reduce the temporary storage required within each block but increase the number of block transitions.

## Implementation

We first import some useful packages

In [ ]:
# Standard library
import logging
import os
from pathlib import Path

# Scientific computing
import numpy as np
import pyvista as pv

# JAX
import jax
import jax.numpy as jnp
import optax

# JAX-FEM
from jax_fem import logger
from jax_fem.generate_mesh import Mesh, box_mesh, get_meshio_cell_type
from jax_fem.problem import Problem
logger.setLevel(logging.WARNING)

jax.config.update("jax_enable_x64", True)
jax.config.update("jax_compiler_enable_remat_pass", False)

print(f"JAX backend: {jax.default_backend()}")

# Directory for the result figures
output_dir = os.path.join("data", "output")
os.makedirs(output_dir, exist_ok=True)

### Mesh

The waveguide has length $L_x=1$ and a square cross-section of width $0.1$. A structured $80\times4\times4$ grid is generated using `jax_fem.generate_mesh.box_mesh` and stored in a `jax_fem.generate_mesh.Mesh` object with trilinear hexahedral elements. The dimensions and material properties use a consistent nondimensional system.

In [ ]:
# Geometry
Lx, Ly, Lz = 1.0, 0.1, 0.1
Nx, Ny, Nz = 80, 4, 4

# Structured hexahedral mesh
ele_type = "HEX8"
cell_type = get_meshio_cell_type(ele_type)
meshio_mesh = box_mesh(Nx, Ny, Nz, Lx, Ly, Lz)
points = meshio_mesh.points
cells = meshio_mesh.cells_dict[cell_type]
mesh = Mesh(points, cells)

# Material properties
E0, nu, rho = 1.0, 0.3, 1.0

print(f"Elements: {mesh.cells.shape[0]}")
print(f"Nodes: {mesh.points.shape[0]}")

In [ ]:
# Visualize the finite element mesh
plotter = pv.Plotter(notebook=True, window_size=(800, 400))
pv_mesh = pv.UnstructuredGrid(
    {getattr(pv.CellType, cell_type.upper()): cells}, points
)
plotter.add_mesh(
    pv_mesh, color="lightgray", show_edges=True, edge_color="black", line_width=0.5,
    ambient=0.45, diffuse=0.7, specular=0.1,
)
span = np.ptp(points, axis=0).max()
origin = points.min(0) - np.array([0.08 * span, 0.08 * span, 0.08 * span])
length = 0.08 * span
for direction, color, label in zip(np.eye(3), ("red", "green", "blue"), ("X", "Y", "Z")):
    plotter.add_mesh(pv.Arrow(start=origin, direction=direction, scale=length), color=color)
    plotter.add_mesh(
        pv.Text3D(label, height=0.025, depth=0.001,
                  center=origin + 1.25 * length * direction, normal=(1, 0, 0)),
        color="black",
    )
plotter.view_isometric(negative=True)
plotter.show(jupyter_backend="html")

### Boundary conditions

The axial displacement is fixed at $x=0$. The normal displacement is fixed on each lateral pair of faces, while the end $x=L_x$ remains traction-free. In JAX-FEM, `dirichlet_bc_info` groups the boundary location functions, constrained displacement components, and prescribed value functions passed to the problem.

In [ ]:
# Surface location functions
left = lambda point: jnp.isclose(point[0], 0.0)
y_sides = lambda point: jnp.isclose(point[1], 0.0) | jnp.isclose(point[1], Ly)
z_sides = lambda point: jnp.isclose(point[2], 0.0) | jnp.isclose(point[2], Lz)
zero = lambda point: 0.0

# JAX-FEM boundary condition helper
dirichlet_bc_info = [
    [left, y_sides, z_sides],
    [0, 1, 2],
    [zero, zero, zero],
]

### Define problem

We define a subclass of `jax_fem.problem.Problem` and implement `get_tensor_map` to provide the linear elastic stress. The Poisson's ratio is fixed, while the layerwise Young's modulus is supplied at the element quadrature points. 

In [ ]:
class LinearElasticity(Problem):
    """
    Small-strain isotropic elasticity with a layerwise Young's modulus.
    """

    # Initialize the fixed Poisson's ratio and quadrature-point modulus
    def custom_init(self, nu):
        self.nu = nu
        self.internal_vars = [jnp.ones((self.num_cells, self.fes[0].num_quads))]

    # Define the stress map used by JAX-FEM
    def get_tensor_map(self):
        def stress(u_grad, E):
            strain = 0.5 * (u_grad + u_grad.T)
            mu = E / (2.0 * (1.0 + self.nu))
            lmbda = E * self.nu / ((1.0 + self.nu) * (1.0 - 2.0 * self.nu))
            return lmbda * jnp.trace(strain) * jnp.eye(3) + 2.0 * mu * strain

        return stress

# Problem instance
problem = LinearElasticity(
    mesh,
    vec=3,
    dim=3,
    ele_type=ele_type,
    dirichlet_bc_info=dirichlet_bc_info,
    additional_info=(nu,),
)

### Lumped mass and initial state

The arrays `fe.shape_vals`, `fe.JxW`, and `fe.cells` provide the shape-function values, weighted Jacobians, and element connectivity used to assemble the row-sum lumped nodal mass $m_a=\int_\Omega\rho N_a\,\mathrm d\Omega$. The first fixed-free longitudinal mode defines the initial displacement.

In [ ]:
# Assemble the row-sum lumped nodal masses
fe = problem.fes[0]
element_mass = rho * jnp.einsum("qi,eq->ei", fe.shape_vals, fe.JxW)
nodal_mass = jnp.zeros(fe.num_total_nodes)
nodal_mass = nodal_mass.at[fe.cells.reshape(-1)].add(element_mass.reshape(-1))
mass = jnp.repeat(nodal_mass[:, None], 3, axis=1)

# Build a mask for the constrained displacement components
fixed = jnp.zeros((fe.num_total_nodes, 3), dtype=bool)
for node_inds, vec_inds in zip(fe.node_inds_list, fe.vec_inds_list):
    fixed = fixed.at[node_inds, vec_inds].set(True)

# First fixed-free longitudinal mode
amplitude = 1.0e-3
wave_number = np.pi / (2.0 * Lx)
mode = jnp.sin(wave_number * fe.points[:, 0])
u0 = jnp.zeros((fe.num_total_nodes, 3)).at[:, 0].set(amplitude * mode)

# Center node of the traction-free end
free_end_node = np.argmin(
    np.linalg.norm(points - np.array([Lx, Ly / 2.0, Lz / 2.0]), axis=1)
)

print(f"Degrees of freedom: {3 * fe.num_total_nodes}")

### Explicit time integration

A fixed time grid spanning one period of the first mode is selected according to the CFL stability condition. At each time step, `problem.compute_residual_vars` assembles the internal force from the current displacement and the quadrature-point modulus. The functions `acceleration` and `step` implement the common operations, and `jax.lax.scan` advances the uniform-material problem while retaining its displacement history.

In [ ]:
# Construct a fixed time grid for one period of the first mode
cfl = 0.4
wave_speed = np.sqrt(E0 * (1.0 - nu) / ((1.0 + nu) * (1.0 - 2.0 * nu) * rho))
period = 2.0 * np.pi / (wave_speed * wave_number)
dt_limit = cfl * min(Lx / Nx, Ly / Ny, Lz / Nz) / wave_speed
num_steps = int(np.ceil(period / dt_limit))
dt = period / num_steps
times = np.arange(num_steps + 1) * dt

print(f"Time steps: {num_steps}")
print(f"Time-step size: {dt:.6e}")

v0 = jnp.zeros_like(u0)

# Compute the unconstrained nodal acceleration
def acceleration(u, E_quad):
    internal_force = problem.compute_residual_vars(
        [u], [E_quad], problem.internal_vars_surfaces
    )[0]
    return jnp.where(fixed, 0.0, -internal_force / mass)

# Advance one central-difference step
def step(carry, E_quad, output_nodes=None):
    u_previous, u = carry
    u_next = 2.0 * u - u_previous + dt**2 * acceleration(u, E_quad)
    u_next = jnp.where(fixed, 0.0, u_next)
    output = u_next if output_nodes is None else u_next[output_nodes, 0]
    return (u, u_next), output

# Integrate the uniform-material problem
E_quad = E0 * jnp.ones((problem.num_cells, fe.num_quads))
u_previous = u0 - dt * v0 + 0.5 * dt**2 * acceleration(u0, E_quad)
u_previous = jnp.where(fixed, 0.0, u_previous)
_, displacement_history = jax.lax.scan(
    lambda carry, _: step(carry, E_quad),
    (u_previous, u0), None, length=num_steps,
)
displacement_history = jnp.concatenate((u0[None, ...], displacement_history))
numerical_signal = displacement_history[:, free_end_node, 0]

### Analytical verification

For the uniform modulus $E_0$, the exact free-end displacement is $u_x(L_x,t)=A\cos(c_p k t)$. This analytical solution is used to verify the numerical response. The numerical wave speed is obtained from the first zero crossing of the mass-weighted modal displacement. The maximum displacement error measures the solution accuracy, while the kinetic, strain, and total energies verify the expected energy exchange and conservation.

In [ ]:
# Analytical free-end response
analytical_signal = amplitude * np.cos(wave_speed * wave_number * times)
numerical_signal = np.asarray(numerical_signal)
normalized_error = np.max(np.abs(numerical_signal - analytical_signal)) / amplitude

# Estimate the wave speed from the mass-weighted modal displacement
modal_signal = np.sum(
    np.asarray(nodal_mass)[None, :] * np.asarray(displacement_history[:, :, 0])
    * np.asarray(mode)[None, :], axis=1
)
crossing = np.flatnonzero(modal_signal[1:] <= 0.0)[0]
fraction = modal_signal[crossing] / (modal_signal[crossing] - modal_signal[crossing + 1])
zero_time = times[crossing] + fraction * dt
measured_speed = (np.pi / (2.0 * zero_time)) / wave_number

# Kinetic, strain, and total energies
velocity = jnp.zeros_like(displacement_history)
velocity = velocity.at[0].set(v0)
velocity = velocity.at[1:-1].set(
    (displacement_history[2:] - displacement_history[:-2]) / (2.0 * dt)
)
velocity = velocity.at[-1].set(
    (3.0 * displacement_history[-1] - 4.0 * displacement_history[-2]
     + displacement_history[-3]) / (2.0 * dt)
)
kinetic_energy = 0.5 * jnp.sum(mass[None, ...] * velocity**2, axis=(1, 2))

u_grad = jnp.einsum("teni,eqnj->teqij", displacement_history[:, fe.cells], fe.shape_grads)
strain = 0.5 * (u_grad + jnp.swapaxes(u_grad, -1, -2))
mu = E0 / (2.0 * (1.0 + nu))
lmbda = E0 * nu / ((1.0 + nu) * (1.0 - 2.0 * nu))
strain_energy_density = (
    0.5 * lmbda * jnp.trace(strain, axis1=-2, axis2=-1)**2
    + mu * jnp.sum(strain**2, axis=(-1, -2))
)
strain_energy = jnp.sum(strain_energy_density * fe.JxW[None, ...], axis=(1, 2))
total_energy = kinetic_energy + strain_energy
energy_drift = jnp.max(jnp.abs(total_energy / total_energy[0] - 1.0))

# Report the forward-solution errors
print(f"Analytical wave speed: {wave_speed:.8f}")
print(f"Measured wave speed:   {measured_speed:.8f}")
print(f"Relative speed error:  {abs(measured_speed / wave_speed - 1.0):.3e}")
print(f"Maximum normalized displacement error: {normalized_error:.3e}")
print(f"Maximum relative total-energy drift: {float(energy_drift):.3e}")

### Synthetic observations

The target Young's modulus $E^\star(x)=E_0(1-0.4x/L_x)$ is represented by one constant value in each axial element layer. Solving the forward problem with $E^\star$ gives the noise-free axial displacement observations at the centerline probes.

In [ ]:
# Assign one modulus parameter to each axial element layer
cell_centers = jnp.mean(fe.points[fe.cells], axis=1)
cell_layer = jnp.clip(jnp.floor(cell_centers[:, 0] * Nx / Lx).astype(int), 0, Nx - 1)
x_layers = (jnp.arange(Nx) + 0.5) * Lx / Nx

# Locate one centerline observation node on each cross-section
x_planes = np.linspace(0.0, Lx, Nx + 1)
probe_nodes = np.array([
    np.argmin(np.linalg.norm(points - [x, Ly / 2.0, Lz / 2.0], axis=1))
    for x in x_planes[1:]
])

# Assemble the forward map required by the inverse problem
def simulate(E, steps=num_steps):
    E_quad = jnp.repeat(E[cell_layer, None], fe.num_quads, axis=1)
    u_previous = u0 - dt * v0 + 0.5 * dt**2 * acceleration(u0, E_quad)
    u_previous = jnp.where(fixed, 0.0, u_previous)
    _, signal = jax.lax.scan(
        lambda carry, _: step(carry, E_quad, probe_nodes),
        (u_previous, u0), None, length=steps,
    )
    return jnp.concatenate((u0[probe_nodes, 0][None, :], signal), axis=0)

# Linear target field
E_target = E0 * (1.0 - 0.4 * x_layers / Lx)

# Generate synthetic observations from the target modulus field
observed_signal = simulate(E_target)
print(f"Material layers: {Nx}")
print(f"Observation points: {probe_nodes.size}")

### Identification

Starting from the uniform field $E=E_0$, MoMo-Adam identifies the $N_x$ layer values by minimizing the normalized displacement mismatch. Reverse-mode AD differentiates the loss through the explicit recurrence. The time steps are grouped into blocks, and `jax.checkpoint` is applied to each block to reduce the intermediate values retained for backpropagation. The results are plotted in the post-processing section below.


In [ ]:
# Checkpointed forward map used during backpropagation
checkpoint_interval = 80
def simulate_checkpointed(E, interval=checkpoint_interval, steps=num_steps):
    # Divide the complete trajectory into equal checkpoint blocks
    if steps % interval != 0:
        raise ValueError("steps must be divisible by interval")

    # Expand the layer values to all element quadrature points
    E_quad = jnp.repeat(E[cell_layer, None], fe.num_quads, axis=1)

    # Initialize the fictitious previous displacement
    u_previous = u0 - dt * v0 + 0.5 * dt**2 * acceleration(u0, E_quad)
    u_previous = jnp.where(fixed, 0.0, u_previous)

    # Advance one block and retain the probe displacements
    def advance_block(carry, _):
        return jax.lax.scan(
            lambda state, _: step(state, E_quad, probe_nodes),
            carry, None, length=interval,
        )

    # Advance the prescribed number of checkpoint blocks
    num_blocks = steps // interval
    _, signal = jax.lax.scan(
        jax.checkpoint(advance_block), (u_previous, u0), None,
        length=num_blocks,
    )

    # Flatten the block dimension and prepend the initial observations
    signal = signal.reshape(steps, probe_nodes.size)
    return jnp.concatenate((u0[probe_nodes, 0][None, :], signal), axis=0)

# Normalized displacement loss and its reverse-mode gradient
scale = jnp.sqrt(jnp.mean(observed_signal**2)) + 1.0e-8
def loss(E):
    prediction = simulate_checkpointed(E)
    return jnp.mean(((prediction - observed_signal) / scale)**2)

value_and_grad = jax.jit(jax.value_and_grad(loss))

In [ ]:
cache_path = Path(output_dir) / "res_explicit_inverse.npz"

# This variable is required whether results are loaded or recomputed.
E_initial = E0 * jnp.ones(Nx)

if cache_path.exists():
    # Load the previously computed inverse results.
    with np.load(cache_path) as cached:
        E_identified = jnp.asarray(cached["E_identified"])
        initial_signal = jnp.asarray(cached["initial_signal"])
        identified_signal = jnp.asarray(cached["identified_signal"])
        loss_history = cached["loss_history"].tolist()
        E_error_history = cached["E_error_history"].tolist()

    print(f"Loaded inverse results from {cache_path}")

else:
    # Adaptive optimizer with the known lower loss bound.
    optimizer = optax.contrib.momo_adam(
        learning_rate=1.0,
        lower_bound=0.0,
        weight_decay=0.0,
    )

    # Uniform initial field and optimizer state.
    E = E_initial
    optimizer_state = optimizer.init(E)
    loss_history = []
    E_error_history = []

    # Record both metrics at the same state:
    # step 0 is the initial field.
    for iteration in range(201):
        value, gradient = value_and_grad(E)
        loss_history.append(float(value))

        E_error = (
            jnp.linalg.norm(E - E_target)
            / jnp.linalg.norm(E_target)
        )
        E_error_history.append(float(E_error))

        if iteration % 20 == 0:
            print(
                f"Step {iteration:02d}: "
                f"loss = {float(value):.3e}, "
                f"E error = {float(E_error):.3e}"
            )

        # Perform 200 updates; step 200 records the final field.
        if iteration < 200:
            updates, optimizer_state = optimizer.update(
                gradient,
                optimizer_state,
                E,
                value=value,
            )
            E = jnp.clip(
                optax.apply_updates(E, updates),
                0.5 * E0,
                E0,
            )

    # Evaluate the initial and identified displacement histories.
    E_identified = E

    identified_signal = simulate(E_identified)
    identified_signal.block_until_ready()

    initial_signal = simulate(E_initial)
    initial_signal.block_until_ready()

    # Save every variable required by the post-processing cell.
    np.savez(
        cache_path,
        E_identified=np.asarray(E_identified),
        initial_signal=np.asarray(initial_signal),
        identified_signal=np.asarray(identified_signal),
        loss_history=np.asarray(loss_history),
        E_error_history=np.asarray(E_error_history),
    )

    print(f"Saved inverse results to {cache_path}")

print(f"Final relative E error: {E_error_history[-1]:.3e}")

### Post-processing

In [ ]:
import matplotlib.pyplot as plt
from matplotlib.ticker import MultipleLocator

# Figure style and colors shared by all figures of this notebook
plot_style = {
    "text.usetex": True, "font.family": "serif", "font.size": 25,
    "axes.labelsize": 25, "xtick.labelsize": 25, "ytick.labelsize": 25,
    "legend.fontsize": 19, "axes.linewidth": 2.0, "axes.grid": True,
    "grid.alpha": 0.22, "grid.linewidth": 0.6, "lines.linewidth": 2.6,
}
blue, red, gray, green, gold = ("#1f77b4", "#d62728", "#7f7f7f", "#008374",
                                "#cf8d2e")
inverse_blue, inverse_red, inverse_gray = "#005F85", "#B92B30", "#454545"
inverse_green, inverse_gold = "#006F5B", "#A85700"
panel_positions = ((0.08, 0.16, 0.375, 0.70), (0.555, 0.16, 0.375, 0.70))

# Free-end histories: the last probe is the center of the traction-free end
observed, initial, identified = (np.asarray(signal)[:, -1] for signal in
                                 (observed_signal, initial_signal, identified_signal))

with plt.rc_context(plot_style):
    fig = plt.figure(figsize=(15, 5.4))
    left, right = (fig.add_axes(position) for position in panel_positions)

    # (a) Free-end displacement histories
    left.plot(times, observed, color=inverse_blue, linewidth=3.0, label="Observed")
    left.plot(times, initial, ":", color=inverse_gray, linewidth=3.0, label="Initial")
    left.plot(times, identified, "--", color=inverse_red, linewidth=3.0, label="Identified")
    left.set(xlabel="Time", ylabel="Free-end displacement",
             ylim=(-1.08e-3, 1.08e-3))
    left.yaxis.set_major_locator(MultipleLocator(5e-4))
    left.ticklabel_format(axis="y", style="sci", scilimits=(0, 0), useMathText=True)
    left.legend(loc="lower left", frameon=False, 
                handlelength=1.5, handletextpad=0.9, 
                borderpad=0.5, labelspacing=0.25, borderaxespad=0.5)

    # (b) loss and modulus error, one curve per axis and no legend
    steps = np.arange(len(loss_history))
    right.semilogy(steps, loss_history, color=inverse_green, linewidth=3.0)
    right.set(xlabel="Optimization step", ylabel=r"Loss $\mathcal{L}$",
              ylim=(1e-8, 1), yticks=np.logspace(-8, 0, 5))
    error_axis = right.twinx()
    error_axis.plot(steps, E_error_history, "--", color=inverse_gold, linewidth=3.0)
    error_axis.set(ylabel=r"Relative error in $E$", ylim=(0.0, 0.30))
    error_axis.yaxis.set_major_locator(MultipleLocator(0.1))
    error_axis.grid(False)
    for ax, color in ((right, inverse_green), (error_axis, inverse_gold)):
        ax.tick_params(axis="y", colors=color)
        ax.yaxis.label.set_color(color)

    # panel labels, offset to the left of the axes
    for ax, tag in zip((left, right), ("(a)", "(b)")):
        ax.text(-0.15, 1.05, tag, transform=ax.transAxes, fontsize=25,
                fontweight="bold", ha="left", va="bottom")

    fig.savefig(os.path.join(output_dir, "res_explicit_inverse.png"), dpi=600)
    plt.show()


### Checkpoint performance

In [ ]:
import json
import psutil
import threading
from time import perf_counter

In [ ]:
# Restart the kernel for each interval
# 0 means no checkpointing
intervals = [800, 1600, 3200, 6400, 0] 
interval = intervals[4]  # Change this value to test different checkpointing intervals
result_path = Path(output_dir) / f"checkpoint_rss_{interval}.json"
if result_path.exists():
    print(f"Using existing result: {result_path}")
else:
    benchmark_steps = 16 * num_steps
    observations = jax.block_until_ready(simulate(E_target, steps=benchmark_steps))
    observation_scale = jax.block_until_ready(jnp.sqrt(jnp.mean(observations**2)) + 1e-8)
    parameters = jax.block_until_ready(E0 * jnp.ones(Nx))

    def benchmark_loss(E, data, scale):
        prediction = (simulate(E, steps=benchmark_steps) if interval == 0 else
                      simulate_checkpointed(E, interval=interval, steps=benchmark_steps))
        return jnp.mean(((prediction - data) / scale)**2)

    # Compile before measuring
    compiled = jax.jit(jax.value_and_grad(benchmark_loss)).lower(
        parameters, observations, observation_scale
    ).compile()
    process = psutil.Process()
    baseline = process.memory_info().rss
    peak = [baseline]
    stop = threading.Event()

    def sample_rss():
        while not stop.is_set():
            peak[0] = max(peak[0], process.memory_info().rss)
            stop.wait(0.002)

    sampler = threading.Thread(target=sample_rss, daemon=True)
    sampler.start()
    try:
        result = jax.block_until_ready(compiled(parameters, observations, observation_scale))
    finally:
        stop.set()
        sampler.join()
        peak[0] = max(peak[0], process.memory_info().rss)

    # Time three subsequent calls. The first call already warmed up.
    samples = []
    for _ in range(3):
        start = perf_counter()
        result = jax.block_until_ready(compiled(parameters, observations, observation_scale))
        samples.append(perf_counter() - start)

    record = {
        "peak_rss_bytes": peak[0],
        "peak_increment_bytes": peak[0] - baseline,
        "runtime_samples_seconds": samples,
    }
    result_path.write_text(json.dumps(record, indent=2))
    print(f"Interval {interval or 'None'}: {peak[0] / 2**30:.2f} GiB, "
          f"{np.median(samples):.2f} s; saved to {result_path}")


In [ ]:
# Load the measured configurations in increasing checkpoint-interval order 
records = [json.loads((Path(output_dir) / f"checkpoint_rss_{i}.json").read_text())
           for i in intervals]
labels = [str(i) if i else "None" for i in intervals]
peak_rss = np.array([r["peak_rss_bytes"] for r in records]) / 2**30
increment = np.array([r["peak_increment_bytes"] for r in records]) / 2**30
samples = np.array([r["runtime_samples_seconds"] for r in records])
runtime = np.median(samples, axis=1)
baseline = peak_rss - increment

with plt.rc_context(plot_style):
    fig = plt.figure(figsize=(15, 5.4))
    memory_ax, runtime_ax = (fig.add_axes(position) for position in panel_positions)
    
    x = np.arange(len(labels))
    bar_style = dict(width=0.4, edgecolor="white", linewidth=0.5)
    memory_ax.bar(x, baseline, color="#E8E5E0",
                  label="Post-compilation baseline", **bar_style)
    memory_bars = memory_ax.bar(x, increment, bottom=baseline, color=blue,
                               label="Execution RSS increment", **bar_style)
    runtime_bars = runtime_ax.bar(
        x, runtime, color=gold, **bar_style,
        # yerr=[runtime - samples.min(axis=1), samples.max(axis=1) - runtime],
        capsize=4, error_kw={"elinewidth": 1.1, "capthick": 1.1})

    memory_ax.set(ylabel="Peak process RSS (GiB)", ylim=(0, 1.45 * peak_rss.max()))
    runtime_ax.set(ylabel="Loss-and-gradient time (s)", ylim=(0, 1.22 * samples.max()))
    memory_ax.bar_label(memory_bars, labels=[f"{v:.2f}" for v in peak_rss],
                        padding=3, fontsize=19)
    runtime_ax.bar_label(runtime_bars, fmt="%.1f", padding=5, fontsize=19)
    memory_ax.legend(loc="upper left", frameon=False, handlelength=1.2, labelspacing=0.3)
    for ax, tag in zip((memory_ax, runtime_ax), ("(a)", "(b)")):
        ax.set(xlabel="Length of checkpoint interval", xticks=x, xticklabels=labels)
        ax.grid(axis="y")
        ax.set_axisbelow(True)
        ax.text(-0.15, 1.05, tag, transform=ax.transAxes, fontsize=25,
                            fontweight="bold", ha="left", va="bottom")
    fig.savefig(os.path.join(output_dir, "res_explicit_checkpoint.png"),
                dpi=600)
    plt.show()
